In [1]:
!pip install yfinance
import pandas as pd
import yfinance as yf
import requests
!pip install lxml

mediummomentum = pd.read_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\mediummomentum.csv", index_col=0, parse_dates=True)
returns_m = pd.read_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\returns_m.csv", index_col=0, parse_dates=True)

In [2]:
mediummomentum.head()

,12M Momentum for AAPL,12M Momentum for AMZN,12M Momentum for AVGO,12M Momentum for GOOG,12M Momentum for GOOGL,12M Momentum for META,12M Momentum for MSFT,12M Momentum for MU,12M Momentum for NVDA,12M Momentum for TSLA
Date,,,,,,,,,,
2010-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
top_list = []
bottom_list = []

for k in range(mediummomentum.shape[0]):
    row = mediummomentum.iloc[k]
    valid = row.dropna()               
    n = max(1, len(valid) // 10)
    order = valid.sort_values() 
    bottom_list.append(order.index[:n].tolist())
    top_list.append(order.index[-n:].tolist())

mediummomentum["top_decile"] = top_list
mediummomentum["bottom_decile"] = bottom_list

In [4]:
stock_cols = [c for c in mediummomentum.columns if c.startswith('12M Momentum for')]
pos = {c: i for i, c in enumerate(stock_cols)}

mediummomentum['top_decile'] = mediummomentum['top_decile'].apply(lambda lst: [pos[c] for c in lst])
mediummomentum['bottom_decile'] = mediummomentum['bottom_decile'].apply(lambda lst: [pos[c] for c in lst])

In [5]:
mediummomentum.tail()

,12M Momentum for AAPL,12M Momentum for AMZN,12M Momentum for AVGO,12M Momentum for GOOG,12M Momentum for GOOGL,12M Momentum for META,12M Momentum for MSFT,12M Momentum for MU,12M Momentum for NVDA,12M Momentum for TSLA,top_decile,bottom_decile
Date,,,,,,,,,,,,
2026-06-30,0.355119,0.292913,0.738391,1.217301,1.248365,-0.052002,-0.109069,4.488015,0.477222,0.101512,[7],[6]
2026-07-31,0.527017,0.233602,0.630083,1.126827,1.163067,-0.141032,-0.087601,6.897155,0.336636,0.371876,[7],[5]
2026-08-31,0.399566,0.018068,0.295600,0.837286,0.867585,-0.269321,-0.295221,9.591579,0.126421,0.364388,[7],[6]
2026-09-30,0.334470,0.185939,0.318579,0.675027,0.677450,-0.243895,-0.077023,5.926737,0.154079,-0.067871,[7],[5]
2026-10-31,0.248951,0.183085,0.128839,0.379870,0.398659,-0.218622,-0.012523,4.739115,0.184809,-0.172625,[7],[5]


In [6]:
returns_m.head()

,AAPL,AMZN,AVGO,GOOG,GOOGL,META,MSFT,MU,NVDA,TSLA
Date,,,,,,,,,,
2010-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-02-28,0.065396,-0.055897,0.044304,-0.005925,-0.005925,NaN,0.022146,0.038990,0.052632,NaN
2010-03-31,0.148470,0.146706,0.132782,0.076538,0.076538,NaN,0.021625,0.144592,0.074074,NaN
2010-04-30,0.111021,0.009796,-0.002432,-0.073036,-0.073036,NaN,0.042677,-0.098360,-0.097127,NaN
2010-05-31,-0.016125,-0.084902,0.006826,-0.076222,-0.076222,NaN,-0.151394,-0.027808,-0.163590,NaN


In [7]:
print(returns_m.tail())

                AAPL      AMZN      AVGO      GOOG     GOOGL      META  \
Date                                                                     
2026-06-30 -0.072742 -0.119347 -0.153148 -0.060801 -0.059832 -0.108612   
2026-07-31  0.067563  0.139465  0.030523  0.009396 -0.003470 -0.011681   
2026-08-31  0.026588 -0.043486 -0.048654 -0.059554 -0.047118  0.028076   
2026-09-30  0.051034 -0.040882 -0.049982  0.016551  0.014590  0.268044   
2026-10-31  0.002012  0.009512  0.011248 -0.001145 -0.001686  0.003999   

                MSFT        MU      NVDA      TSLA  
Date                                                
2026-06-30 -0.171509  0.188764 -0.051231 -0.034856  
2026-07-31  0.245831 -0.286872  0.003299 -0.260081  
2026-08-31  0.093659  0.164878  0.099776  0.182321  
2026-09-30  0.011059  0.110959  0.035581 -0.035711  
2026-10-31  0.009027  0.009182  0.024389  0.044475  


In [12]:
def portfolio(k):
    top_returns = []
    bottom_returns = []

    for i in range(mediummomentum.shape[0]):
        top = mediummomentum["top_decile"].iloc[i]
        bottom = mediummomentum["bottom_decile"].iloc[i]

        if i + k > returns_m.shape[0] or len(top) == 0 or len(bottom) == 0:
            top_returns.append(None)
            bottom_returns.append(None)
        else:
            total = 0
            for j in range(len(top)):
                growth = 1
                for o in range(k):
                    growth *= (1 + returns_m.iloc[i+o, top[j]])
                total = total + growth -1
            top_returns.append(total/len(top))

            total = 0
            for j in range(len(bottom)):
                growth = 1
                for o in range(k):
                    growth = growth * (1 + returns_m.iloc[i + o, bottom[j]])
                total = total + (growth - 1)
            bottom_returns.append(total / len(bottom))

    results = pd.DataFrame(index = mediummomentum.index)
    results["top"] = top_returns
    results["bottom"] = bottom_returns
    results["long_short"] = results["top"] - results["bottom"]
    return results


portfolio(1).head(25)
 


,top,bottom,long_short
Date,,,
2010-01-31,NaN,NaN,NaN
2010-02-28,NaN,NaN,NaN
2010-03-31,NaN,NaN,NaN
2010-04-30,NaN,NaN,NaN
2010-05-31,NaN,NaN,NaN
2010-06-30,NaN,NaN,NaN
2010-07-31,NaN,NaN,NaN
2010-08-31,NaN,NaN,NaN
2010-09-30,NaN,NaN,NaN
